# 03 — Simulator validation: real vs simulated stylized facts

The headline artifact: the same battery on (a) the real fixture and (b) trades exported from the Rust Hawkes simulator (`quant-sim run scenarios/hawkes_demo.toml` + `export-events`). Preregistered expected failures of the v1 exp-kernel simulator, stated before results: long-memory of signs, volatility roughness, Zumbach asymmetry.

In [ ]:
FIXTURE_ONLY = True  # papermill parameter

In [ ]:
import numpy as np
import polars as pl
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from quantsim_research.config import repo_root

FIXTURE = repo_root() / "data" / "fixtures" / "binance-um" / "trades" / "BTCUSDT-2024-01-07-h00.csv"
ARTIFACTS = repo_root() / "artifacts" / "notebooks"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
df = pl.read_csv(FIXTURE)
ts_ns = df["ts_ns"].to_numpy()
price = df["price_e8"].to_numpy() / 1e8
side = df["side"].to_numpy()
print(f"fixture: {len(df)} trades, {(ts_ns[-1]-ts_ns[0])/1e9:.0f}s span")


In [ ]:
from quantsim_research.stylized_facts.autocorr import acf, ljung_box
from quantsim_research.stylized_facts.orderflow import trade_sign_autocorr, zumbach_asymmetry
from quantsim_research.stylized_facts.returns import moments


def battery(ts, px, sd, label):
    sec = ((ts - ts[0]) / 1e9).astype(int)
    last = {}
    for s, p in zip(sec, px, strict=True):
        last[s] = p
    grid = np.array([last[k] for k in sorted(last)])
    r = np.diff(np.log(grid))
    m = moments(r)
    rho_abs = acf(np.abs(r), 20)
    sacf = trade_sign_autocorr(sd.astype(float), max_lag=50)
    z = zumbach_asymmetry(grid, window=30)
    return {
        "label": label,
        "ex_kurt": round(m.excess_kurtosis, 2),
        "ljung_absr_p": f"{ljung_box(np.abs(r), 10).p_value:.1e}",
        "acf_abs_lag1": round(float(rho_abs[0]), 3),
        "sign_acf_lag1": round(float(sacf[0]), 3),
        "sign_acf_lag20": round(float(sacf[19]), 3),
        "zumbach": round(z.asymmetry, 3) if np.isfinite(z.asymmetry) else None,
    }


rows = [battery(ts_ns, price, side, "real BTCUSDT (1h)")]


In [ ]:
# Simulated trades, if a Rust export is present (guarded so the notebook
# always runs offline; CI generates the export before executing).
sim_csv = repo_root() / "runs" / "hawkes_demo" / "trades.csv"
if sim_csv.exists():
    sim = pl.read_csv(sim_csv)
    rows.append(
        battery(
            sim["ts_ns"].to_numpy(),
            sim["price_e8"].to_numpy() / 1e8,
            sim["side"].to_numpy(),
            "sim exp-Hawkes",
        )
    )
else:
    print("no sim export found - run `quant-sim run scenarios/hawkes_demo.toml` "
          "and `quant-sim export-events` first for the full comparison")
table = pl.DataFrame(rows)
print(table)
table.write_csv(ARTIFACTS / "03_real_vs_sim.csv")


### Reading the table
- **Volatility clustering / fat tails**: exp-Hawkes partially reproduces (clustered arrivals ⇒ clustered activity).
- **Sign long-memory**: expected shortfall — sim sign ACF decays exponentially, real decays as a power law (order splitting).
- **Zumbach**: expected ≈ 0 in sim (linear Hawkes is TR-symmetric); real is positive. Stage 2's quadratic Hawkes targets exactly this.